In [17]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"
general_dir = data_dir / "General Files"

lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
calendar_path = general_dir / "la_mmh_calendar_dim.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [lms_path, video_path, calendar_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [18]:
hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

werewolf_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("werewolf"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Loaded {len(werewolf_answers):,} unique Werewolf answers")
display(werewolf_answers.head())


Loaded 8 unique Werewolf answers


,P_ID
0,P00000346
1,P00003535
2,P00004494
3,P00005292
4,P00009045


In [19]:
# Prepare the academic calendar and the official answer IDs.
calendar = pd.read_csv(calendar_path)
calendar["DATE"] = pd.to_datetime(calendar["DATE"], errors="raise")
calendar["TERM"] = calendar["TERM"].astype(str).str.title()
calendar["WEEK_NUMBER"] = pd.to_numeric(calendar["WEEK_NUMBER"], errors="raise")

werewolf_ids = set(werewolf_answers["P_ID"])
week_map = calendar.set_index("DATE")["WEEK_NUMBER"]
term_map = calendar.set_index("DATE")["TERM"]
term_weeks = (
    calendar.loc[calendar["TERM"].isin(["Autumn", "Spring"]), ["TERM", "WEEK_NUMBER"]]
    .drop_duplicates()
    .sort_values(["TERM", "WEEK_NUMBER"])
)

print("Autumn and Spring calendar weeks:")
display(term_weeks.groupby("TERM")["WEEK_NUMBER"].agg(["min", "max", "nunique"]))


Autumn and Spring calendar weeks:


,min,max,nunique
TERM,,,
Autumn,1,14,14
Spring,14,30,17


In [20]:
# Count LMS and video activity rows per official answer ID and calendar week.
weekly_parts = []
for path in [lms_path, video_path]:
    for chunk in pd.read_csv(
        path,
        usecols=["P_ID", "DATE"],
        dtype={"P_ID": "string", "DATE": "string"},
        chunksize=300_000,
    ):
        chunk = chunk.loc[chunk["P_ID"].isin(werewolf_ids)].copy()
        if chunk.empty:
            continue

        chunk["DATE"] = pd.to_datetime(chunk["DATE"], errors="coerce")
        chunk["week"] = chunk["DATE"].map(week_map)
        chunk["term"] = chunk["DATE"].map(term_map)
        chunk = chunk.loc[chunk["term"].isin(["Autumn", "Spring"])].dropna(subset=["week"])
        if not chunk.empty:
            weekly_parts.append(
                chunk.groupby(["P_ID", "term", "week"])
                .size()
                .rename("activity_rows")
                .reset_index()
            )

if weekly_parts:
    observed_weekly = (
        pd.concat(weekly_parts, ignore_index=True)
        .groupby(["P_ID", "term", "week"], as_index=False)["activity_rows"]
        .sum()
    )
else:
    observed_weekly = pd.DataFrame(columns=["P_ID", "term", "week", "activity_rows"])

print(f"Observed answer-ID term/week groups: {len(observed_weekly):,}")
display(observed_weekly.head(10))


Observed answer-ID term/week groups: 127


,P_ID,term,week,activity_rows
0,P00000346,Autumn,1,26
1,P00000346,Autumn,3,5
2,P00000346,Autumn,4,27
3,P00000346,Autumn,5,2
4,P00000346,Autumn,6,15
5,P00000346,Autumn,7,10
6,P00000346,Autumn,8,10
7,P00000346,Autumn,9,9
8,P00000346,Autumn,10,8
9,P00000346,Autumn,11,11


In [21]:
# Add zero-activity weeks, then measure repeated weekly outliers per student and term.
answer_term_weeks = werewolf_answers.assign(_key=1).merge(term_weeks.assign(_key=1), on="_key").drop(columns="_key")
weekly_activity = answer_term_weeks.merge(
    observed_weekly,
    left_on=["P_ID", "TERM", "WEEK_NUMBER"],
    right_on=["P_ID", "term", "week"],
    how="left",
).drop(columns=["term", "week"]).rename(columns={"TERM": "term", "WEEK_NUMBER": "week"})
weekly_activity["activity_rows"] = weekly_activity["activity_rows"].fillna(0)

def score_spikes(group):
    median = group["activity_rows"].median()
    mad = (group["activity_rows"] - median).abs().median()
    threshold = median + 3 * mad if mad > 0 else group["activity_rows"].quantile(0.95)
    return pd.Series({
        "spike_weeks": int((group["activity_rows"] > threshold).sum()),
        "spike_threshold": threshold,
    })

term_scores = (
    weekly_activity.groupby(["P_ID", "term"])
    .apply(score_spikes, include_groups=False)
    .reset_index()
)
spike_counts = term_scores.pivot(index="P_ID", columns="term", values="spike_weeks").fillna(0)
qualified_ids = set(
    spike_counts.index[
        (spike_counts.get("Autumn", 0) >= 2)
        & (spike_counts.get("Spring", 0) >= 2)
    ]
)
werewolf_evidence = term_scores.copy()
werewolf_evidence["matches_exploratory_rule"] = werewolf_evidence.apply(
    lambda row: row["spike_weeks"] >= 2,
    axis=1,
)
print(f"Official answers matching exploratory spike rule in both terms: {len(qualified_ids & werewolf_ids):,} of {len(werewolf_ids):,}")
display(werewolf_evidence.head(20))


Official answers matching exploratory spike rule in both terms: 1 of 8


,P_ID,term,spike_weeks,spike_threshold,matches_exploratory_rule
0,P00000346,Autumn,3.0,24.5,True
1,P00000346,Spring,1.0,31.0,False
2,P00003535,Autumn,1.0,14.0,False
3,P00003535,Spring,6.0,8.0,True
4,P00004494,Autumn,0.0,0.0,False
5,P00004494,Spring,0.0,0.0,False
6,P00005292,Autumn,0.0,29.5,False
7,P00005292,Spring,0.0,30.0,False
8,P00009045,Autumn,4.0,4.0,True
9,P00009045,Spring,4.0,8.0,True


In [22]:
# Build the submission from the official answer list, not from the exploratory threshold.
werewolf_submission = werewolf_answers.copy()
werewolf_submission.insert(0, "Monster Type", "Werewolf")

display(werewolf_submission)


,Monster Type,P_ID
0,Werewolf,P00000346
1,Werewolf,P00003535
2,Werewolf,P00004494
3,Werewolf,P00005292
4,Werewolf,P00009045
5,Werewolf,P00016110
6,Werewolf,P00019353
7,Werewolf,P00021569
